# SpendShield v2 anomaly detection evaluation

This experiment fits a dependency-free robust-MAD anomaly model on the v2 training split only. It evaluates ranking against synthetic scenario labels offline; it is not real-world fraud detection, a production score, or a payment decision.

Success criteria: no label or future-data leakage, deterministic outputs, documented score direction, train-only thresholds, and complete validation/test ranking artifacts.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'ml').exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))
FEATURE_DIR = ROOT / 'data' / 'synthetic' / 'v2' / 'features'
ARTIFACT_DIR = ROOT / 'data' / 'synthetic' / 'anomaly_detection' / 'v2'
from ml.anomaly_detection import build_anomaly_artifact, validate_anomaly_artifact

SEED = 20260913
FEATURE_DIR, ARTIFACT_DIR

(WindowsPath('D:/E_Drive/Project/data/synthetic/v2/features'),
 WindowsPath('D:/E_Drive/Project/data/synthetic/anomaly_detection/v2'))

## Model configuration and leakage boundary

The model uses 12 numeric/prior-only v2 features. Identifiers, split assignment, scenario labels, generator metadata, and future/post-event fields are excluded. The robust scale is `max(1.4826 * MAD, IQR / 1.349, 1e-9)`, and per-feature distance is capped at 8 before averaging.

In [2]:
summary = build_anomaly_artifact(FEATURE_DIR, ARTIFACT_DIR)
model_config = json.loads((ARTIFACT_DIR / 'anomaly_model_config.json').read_text(encoding='utf-8'))
feature_manifest = json.loads((ARTIFACT_DIR / 'anomaly_feature_manifest.json').read_text(encoding='utf-8'))
{
    'dataset_version': summary['dataset']['version'],
    'feature_version': summary['dataset']['feature_version'],
    'model_type': model_config['model_type'],
    'feature_count': len(model_config['feature_names']),
    'fit_data': model_config['fit_data'],
    'labels_used_during_fit': model_config['labels_used_during_fit'],
    'scenario_metadata_used_during_fit': model_config['scenario_metadata_used_during_fit'],
    'score_semantics': model_config['score_semantics'],
    'leakage_valid': feature_manifest['leakage_review']['valid'],
}

{'dataset_version': 'v2',
 'feature_version': '1.1.0',
 'model_type': 'robust_mad_distance',
 'feature_count': 12,
 'fit_data': 'train split only',
 'labels_used_during_fit': False,
 'scenario_metadata_used_during_fit': False,
 'score_semantics': {'anomaly_score': 'raw score min-max normalized with train raw min/max and clipped to [0, 1]',
  'comparability': 'Comparable only within this dataset/model configuration; not comparable across versions without a controlled recalibration study.',
  'direction': 'higher means more anomalous',
  'raw_score': 'mean capped robust distance from train-fitted feature medians; higher means more anomalous'},
 'leakage_valid': True}

## Validation and test analysis

The tables below are synthetic-label ranking analyses only. Top-k composition, normal-class representation, scenario coverage, and threshold recall must not be interpreted as real-world performance. Thresholds are fixed from training score percentiles and are not decision thresholds.

In [3]:
evaluation_view = {}
for split in ('validation', 'test'):
    split_result = summary['evaluation'][split]
    evaluation_view[split] = {
        'rows': split_result['row_count'],
        'score_distribution': split_result['score_distribution'],
        'synthetic_only_ranking_metrics': split_result['synthetic_only_ranking_metrics'],
        'top_1_percent': split_result['top_k_composition']['top_1_percent'],
        'top_5_percent': split_result['top_k_composition']['top_5_percent'],
        'top_10_percent': split_result['top_k_composition']['top_10_percent'],
    }
evaluation_view

{'validation': {'rows': 1474,
  'score_distribution': {'count': 1474,
   'minimum': 0.070426,
   'maximum': 0.885543,
   'mean': 0.234781,
   'median': 0.210498,
   'p95': 0.45767,
   'p99': 0.585794},
  'synthetic_only_ranking_metrics': {'target_definition': 'normal versus any non-normal synthetic scenario',
   'synthetic_label_evaluation_only': True,
   'roc_auc': 0.562733,
   'average_precision': 0.417902,
   'limitation': 'These are synthetic-label ranking analyses, not real-world detection metrics.'},
  'top_1_percent': {'fraction': 0.01,
   'top_k_count': 15,
   'class_counts': {'normal': 4,
    'synthetic_behavior_deviation': 3,
    'synthetic_combined_pattern': 0,
    'synthetic_high_amount': 4,
    'synthetic_rapid_repeat': 4,
    'synthetic_unusual_time': 0},
   'class_composition': {'normal': 0.266667,
    'synthetic_behavior_deviation': 0.2,
    'synthetic_combined_pattern': 0.0,
    'synthetic_high_amount': 0.266667,
    'synthetic_rapid_repeat': 0.266667,
    'synthetic_u

## Validation and limitations

The phase is complete only when artifacts, leakage checks, reproducibility, and notebook execution pass. Synthetic labels are not real ground truth; normal and synthetic patterns overlap; the model is not exported for production inference; and no backend, database, payment, or notification behavior is created.

In [4]:
artifact_validation = validate_anomaly_artifact(ARTIFACT_DIR)
{
    'artifact_valid': artifact_validation['valid'],
    'normalized_scores_in_range': artifact_validation['normalized_scores_in_range'],
    'decision': 'ANOMALY_EVALUATION_READY_FOR_BACKEND_RESEARCH_INTEGRATION' if artifact_validation['valid'] else 'ANOMALY_EVALUATION_REQUIRES_REVIEW',
    'limitations': summary['limitations'],
}

{'artifact_valid': True,
 'normalized_scores_in_range': True,
 'decision': 'ANOMALY_EVALUATION_READY_FOR_BACKEND_RESEARCH_INTEGRATION',
 'limitations': ['Synthetic scenario labels are not real ground truth.',
  'Synthetic patterns may be easier to rank than real behavior.',
  'Normal and synthetic patterns overlap, so ranking is not certainty.',
  'Thresholds are research percentiles, not decisions.',
  'Explanations are deterministic signal summaries, not causal explanations.',
  'No production inference or backend/database integration was implemented.']}